In [3]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
from astropy.time import Time
import astropy.units as u

# Protocol 1

In [4]:
# ---------------------------------------------------
# 1. Observatory location and observation time
# ---------------------------------------------------
lat =  51.459110 # -32.9685 -> Darren; 51.459110 -> Bochum
lon = 7.242801   # 151.6601 -> Darren; 7.242801 -> Bochum
height_m = 12

location = EarthLocation(
    lat=lat * u.deg,
    lon=lon * u.deg,
    height=height_m * u.m
)

t_obs = Time("2026-08-10T23:00:00", scale="utc")
unix_ts = int(t_obs.unix)


# ---------------------------------------------------
# 2. Define Alt/Az scan grid
# ---------------------------------------------------
altitudes_deg = [15, 30, 45, 60, 75]
az_list = np.arange(0, 360, 45)
#az_list = [20, 40]

# ---------------------------------------------------
# 3. Create AltAz frame
# ---------------------------------------------------
altaz_frame = AltAz(obstime=t_obs, location=location)

# ---------------------------------------------------
# 4. Convert Alt/Az → RA/Dec
# ---------------------------------------------------
pointings = []

for alt in altitudes_deg:
    for az in az_list:

        altaz_coord = SkyCoord(
            alt=alt * u.deg,
            az=az * u.deg,
            frame=altaz_frame
        )

        icrs_coord = altaz_coord.transform_to("icrs")

        pointings.append({
            "alt_deg": alt,
            "az_deg": az,
            "ra_deg": icrs_coord.ra.deg,
            "dec_deg": icrs_coord.dec.deg
        })

# ---------------------------------------------------
# 5. Convert to DataFrame
# ---------------------------------------------------
df_pointings = pd.DataFrame(pointings)

# ---------------------------------------------------
# 6. Print results
# ---------------------------------------------------
for _, row in df_pointings.iterrows():
    print(
        f"Alt={row.alt_deg:5.1f}  "
        f"Az={row.az_deg:6.1f}  →  "
        f"RA={row.ra_deg:8.3f}  "
        f"Dec={row.dec_deg:8.3f}"
    )

Alt= 15.0  Az=   0.0  →  RA= 131.191  Dec=  53.637
Alt= 15.0  Az=  45.0  →  RA=  69.863  Dec=  38.852
Alt= 15.0  Az=  90.0  →  RA=  31.840  Dec=  11.551
Alt= 15.0  Az= 135.0  →  RA= 355.810  Dec= -13.044
Alt= 15.0  Az= 180.0  →  RA= 311.280  Dec= -23.639
Alt= 15.0  Az= 225.0  →  RA= 266.816  Dec= -12.883
Alt= 15.0  Az= 270.0  →  RA= 230.837  Dec=  11.773
Alt= 15.0  Az= 315.0  →  RA= 192.727  Dec=  39.045
Alt= 30.0  Az=   0.0  →  RA= 131.061  Dec=  68.638
Alt= 30.0  Az=  45.0  →  RA=  56.479  Dec=  50.510
Alt= 30.0  Az=  90.0  →  RA=  21.525  Dec=  22.883
Alt= 30.0  Az= 135.0  →  RA= 349.094  Dec=   0.398
Alt= 30.0  Az= 180.0  →  RA= 311.313  Dec=  -8.640
Alt= 30.0  Az= 225.0  →  RA= 273.570  Dec=   0.537
Alt= 30.0  Az= 270.0  →  RA= 241.174  Dec=  23.092
Alt= 30.0  Az= 315.0  →  RA= 206.124  Dec=  50.721
Alt= 45.0  Az=   0.0  →  RA= 130.368  Dec=  83.637
Alt= 45.0  Az=  45.0  →  RA=  35.533  Dec=  59.720
Alt= 45.0  Az=  90.0  →  RA=   9.391  Dec=  33.432
Alt= 45.0  Az= 135.0  →  RA= 34

In [5]:
base_tail = f"c=3970&et=3970&g=20&d=600&t={unix_ts}"

# Group pointings by altitude (in creation order)
from collections import defaultdict
by_alt = defaultdict(list)
for p in pointings:
    by_alt[p["alt_deg"]].append(p)

all_links = []
counter = 1

for alt in sorted(by_alt.keys()):
    print(f"* {int(alt)} degrees")

    ring = by_alt[alt]
    ring8 = ring[:8] + [ring[0]]

    for p in ring8:
        group_id = f"x20260407skyglow{counter:05d}"
        link = (
            f"unistellar://science/transient?"
            f"ra={p['ra_deg']:.6f}&dec={p['dec_deg']:.6f}&{base_tail}&scitag={group_id}"
        )

        test_name = f"Test {counter}_alt_{int(alt)}"

        # HTML block for every link
        print('[[html]]')
        print(f'<a href="{link}"> {test_name} </a>')
        print('[[/html]]')
        print(f"{test_name}: {link}  (valid at {t_obs.iso} UTC)")

        all_links.append(link)
        counter += 1


* 15 degrees
[[html]]
<a href="unistellar://science/transient?ra=131.190745&dec=53.637287&c=3970&et=3970&g=20&d=600&t=1786402800&scitag=x20260407skyglow00001"> Test 1_alt_15 </a>
[[/html]]
Test 1_alt_15: unistellar://science/transient?ra=131.190745&dec=53.637287&c=3970&et=3970&g=20&d=600&t=1786402800&scitag=x20260407skyglow00001  (valid at 2026-08-10 23:00:00.000 UTC)
[[html]]
<a href="unistellar://science/transient?ra=69.862846&dec=38.851938&c=3970&et=3970&g=20&d=600&t=1786402800&scitag=x20260407skyglow00002"> Test 2_alt_15 </a>
[[/html]]
Test 2_alt_15: unistellar://science/transient?ra=69.862846&dec=38.851938&c=3970&et=3970&g=20&d=600&t=1786402800&scitag=x20260407skyglow00002  (valid at 2026-08-10 23:00:00.000 UTC)
[[html]]
<a href="unistellar://science/transient?ra=31.839987&dec=11.551261&c=3970&et=3970&g=20&d=600&t=1786402800&scitag=x20260407skyglow00003"> Test 3_alt_15 </a>
[[/html]]
Test 3_alt_15: unistellar://science/transient?ra=31.839987&dec=11.551261&c=3970&et=3970&g=20&d=600

In [6]:
for _, row in df_pointings.iterrows():
    print(
        f"Alt={row.alt_deg:5.1f}  Az={row.az_deg:6.1f}  →  "
        f"RA={row.ra_deg:.6f}  Dec={row.dec_deg:.6f}"
    )

Alt= 15.0  Az=   0.0  →  RA=131.190745  Dec=53.637287
Alt= 15.0  Az=  45.0  →  RA=69.862846  Dec=38.851938
Alt= 15.0  Az=  90.0  →  RA=31.839987  Dec=11.551261
Alt= 15.0  Az= 135.0  →  RA=355.809591  Dec=-13.043952
Alt= 15.0  Az= 180.0  →  RA=311.280036  Dec=-23.639376
Alt= 15.0  Az= 225.0  →  RA=266.816341  Dec=-12.883440
Alt= 15.0  Az= 270.0  →  RA=230.836954  Dec=11.772598
Alt= 15.0  Az= 315.0  →  RA=192.727249  Dec=39.044985
Alt= 30.0  Az=   0.0  →  RA=131.060680  Dec=68.637605
Alt= 30.0  Az=  45.0  →  RA=56.479300  Dec=50.510295
Alt= 30.0  Az=  90.0  →  RA=21.525269  Dec=22.882693
Alt= 30.0  Az= 135.0  →  RA=349.094392  Dec=0.398124
Alt= 30.0  Az= 180.0  →  RA=311.312845  Dec=-8.639641
Alt= 30.0  Az= 225.0  →  RA=273.569575  Dec=0.536836
Alt= 30.0  Az= 270.0  →  RA=241.174396  Dec=23.091893
Alt= 30.0  Az= 315.0  →  RA=206.123933  Dec=50.720932
Alt= 45.0  Az=   0.0  →  RA=130.368130  Dec=83.637407
Alt= 45.0  Az=  45.0  →  RA=35.533348  Dec=59.719503
Alt= 45.0  Az=  90.0  →  RA=9.39

# Protocol 2

In [11]:
# ---------------------------------------------------
# 1. Observatory location
# ---------------------------------------------------
lat = 51.459110      # Bochum
lon = 7.242801
height_m = 12
 
location = EarthLocation(lat=lat * u.deg, lon=lon * u.deg, height=height_m * u.m)
 
# ---------------------------------------------------
# 2. Observation time
#    - Use Time.now() to generate links for RIGHT NOW
#    - Or set a fixed future time if scheduling ahead
#      (make sure you open the links close to this time!)
# ---------------------------------------------------
t_obs = Time("2026-08-11T22:15:00", scale="utc")  # <-- edit if scheduling ahead
 
print(f"Observation time (UTC): {t_obs.iso}")
 
# ---------------------------------------------------
# 3. Define your Alt/Az targets
#    (edit this list to whatever grid/ring you need)
# ---------------------------------------------------
altaz_targets = [
    (70.9,   0.0), (70.9,  90.0), (70.9, 180.0), (70.9, 270.0),
    (48.2, 315.0), (48.2, 225.0), (48.2, 135.0), (48.2,  45.0),
    (43.9,   0.0), (43.9,  90.0), (43.9, 180.0), (43.9, 270.0),
    (24.1, 337.5), (24.1, 292.5), (24.1, 247.5), (24.1, 202.5),
    (24.1, 157.5), (24.1, 112.5), (24.1,  67.5), (24.1,  22.5),
    (16.3,  45.0), (16.3, 135.0), (16.3, 225.0), (16.3, 315.0),
]
 
# ---------------------------------------------------
# 4. Convert Alt/Az -> RA/Dec for t_obs
# ---------------------------------------------------
altaz_frame = AltAz(obstime=t_obs, location=location)
 
pointings = []
for alt, az in altaz_targets:
    altaz_coord = SkyCoord(alt=alt * u.deg, az=az * u.deg, frame=altaz_frame)
    icrs_coord = altaz_coord.transform_to("icrs")
    pointings.append({
        "alt_deg": alt,
        "az_deg": az,
        "ra_deg": icrs_coord.ra.deg,
        "dec_deg": icrs_coord.dec.deg,
    })
 
df_pointings = pd.DataFrame(pointings)
 
print("\nComputed pointings:")
for _, row in df_pointings.iterrows():
    print(
        f"Alt={row.alt_deg:5.1f}  Az={row.az_deg:6.1f}  ->  "
        f"RA={row.ra_deg:8.3f}  Dec={row.dec_deg:8.3f}"
    )
 
# ---------------------------------------------------
# 5. Build deep-links, synced to the same t_obs
# ---------------------------------------------------
unix_ts = int(t_obs.unix)
base_tail = f"c=3970&et=3970&g=20&d=120&t={unix_ts}"
 
by_alt = defaultdict(list)
for p in pointings:
    by_alt[p["alt_deg"]].append(p)
 
all_links = []
counter = 1
 
 
for alt in sorted(by_alt.keys(), reverse=True):
    print(f"* {alt} degrees")
    for p in by_alt[alt]:
        group_id = f"x20260407skyglow{counter:05d}"
        link = (
            f"unistellar://science/transient?"
            f"ra={p['ra_deg']:.6f}&dec={p['dec_deg']:.6f}&{base_tail}&scitag={group_id}"
        )
        test_name = f"Test {counter}_alt_{int(round(alt))}"
 
        print('[[html]]')
        print(f'<a href="{link}"> {test_name} </a>')
        print('[[/html]]')
 
        all_links.append(link)
        counter += 1
 
print(f"\n# Total links: {len(all_links)}")
print(f"# Open these at/near {t_obs.iso} UTC -- targets drift ~15 deg/hour otherwise.")

Observation time (UTC): 2026-08-11 22:15:00.000

Computed pointings:
Alt= 70.9  Az=   0.0  ->  RA= 301.388  Dec=  70.481
Alt= 70.9  Az=  90.0  ->  RA= 330.179  Dec=  47.526
Alt= 70.9  Az= 180.0  ->  RA= 301.116  Dec=  32.281
Alt= 70.9  Az= 270.0  ->  RA= 272.135  Dec=  47.648
Alt= 48.2  Az= 315.0  ->  RA= 222.718  Dec=  61.358
Alt= 48.2  Az= 225.0  ->  RA= 271.584  Dec=  16.819
Alt= 48.2  Az= 135.0  ->  RA= 330.555  Dec=  16.693
Alt= 48.2  Az=  45.0  ->  RA=  19.453  Dec=  61.114
Alt= 43.9  Az=   0.0  ->  RA= 120.099  Dec=  82.516
Alt= 43.9  Az=  90.0  ->  RA=   0.090  Dec=  32.695
Alt= 43.9  Az= 180.0  ->  RA= 301.047  Dec=   5.282
Alt= 43.9  Az= 270.0  ->  RA= 242.070  Dec=  32.911
Alt= 24.1  Az= 337.5  ->  RA= 161.736  Dec=  57.794
Alt= 24.1  Az= 292.5  ->  RA= 209.811  Dec=  32.608
Alt= 24.1  Az= 247.5  ->  RA= 243.085  Dec=   5.906
Alt= 24.1  Az= 202.5  ->  RA= 280.089  Dec= -11.918
Alt= 24.1  Az= 157.5  ->  RA= 321.930  Dec= -12.011
Alt= 24.1  Az= 112.5  ->  RA= 359.004  Dec=   5

In [9]:
import zoneinfo
from datetime import timezone
local_tz = zoneinfo.ZoneInfo("Europe/Berlin")
print(f"Observation time: {t_obs.iso} UTC  |  {t_obs.to_datetime(timezone.utc).astimezone(local_tz)} local")

Observation time: 2026-08-11 21:00:00.000 UTC  |  2026-08-11 23:00:00+02:00 local
